In [2]:
import os
import torch
import torchaudio
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from transformers import Wav2Vec2Processor, HubertModel
from sklearn.preprocessing import LabelEncoder
from pathlib import Path
from tqdm import tqdm

/opt/miniconda3/envs/torchpy311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Training HuBERT on synthetic fan data generated from GAN

In [2]:
# ----------------------------
# Config
# ----------------------------
class Config:
    SAMPLE_RATE = 16000
    MODEL_NAME = "facebook/hubert-base-ls960"
    BATCH_SIZE = 4
    EPOCHS = 50
    LR = 1e-3
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    DATA_PATH = Path("generated_samples_fan/final_samples")

config = Config()

# ----------------------------
# Dataset
# ----------------------------
class AudioDataset(Dataset):
    def __init__(self, data_path):
        self.filepaths = []
        self.labels = []

        for label in ["normal", "abnormal"]:
            folder = data_path / label
            for file in folder.glob("*.wav"):
                self.filepaths.append(file)
                self.labels.append(label)

        self.label_encoder = LabelEncoder()
        self.encoded_labels = self.label_encoder.fit_transform(self.labels)

        # self.processor = Wav2Vec2Processor.from_pretrained(config.MODEL_NAME)

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        path = self.filepaths[idx]
        waveform, sr = torchaudio.load(path)
        waveform = torchaudio.functional.resample(waveform, sr, config.SAMPLE_RATE)
        # input_values = self.processor(waveform.squeeze().numpy(), sampling_rate=config.SAMPLE_RATE, return_tensors="pt").input_values.squeeze(0)
        input_values = waveform.squeeze(0)  # Use raw waveform for HuBERT
        label = torch.tensor(self.encoded_labels[idx], dtype=torch.long)
        return input_values, label

In [ ]:
# ----------------------------
# Transformer Model
# ----------------------------
class HubertClassifier(nn.Module):
    def __init__(self, hidden_dim=768, num_classes=2):
        super(HubertClassifier, self).__init__()
        self.hubert = HubertModel.from_pretrained(config.MODEL_NAME)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes)
        )

    def forward(self, input_values):
        with torch.no_grad():
            features = self.hubert(input_values).last_hidden_state
        pooled = torch.mean(features, dim=1)
        logits = self.classifier(pooled)
        return logits

# ----------------------------
# Training Loop
# ----------------------------
def train():
    dataset = AudioDataset(config.DATA_PATH)
    dataloader = DataLoader(dataset, batch_size=config.BATCH_SIZE, shuffle=True)

    model = HubertClassifier().to(config.DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.classifier.parameters(), lr=config.LR)

    model.train()
    for epoch in range(config.EPOCHS):
        running_loss = 0.0
        for inputs, labels in tqdm(dataloader):
            inputs = inputs.to(config.DEVICE)
            labels = labels.to(config.DEVICE)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f"Epoch {epoch + 1}/{config.EPOCHS}, Loss: {running_loss:.4f}")

    torch.save(model.state_dict(), "hubert_transformer_synthetic.pth")

# ----------------------------
# Run
# ----------------------------
if __name__ == '__main__':
    train()

config.json:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

  0%|          | 0/100 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

100%|██████████| 100/100 [03:00<00:00,  1.81s/it]


...

Epoch 48/50, Loss: 69.3295


100%|██████████| 100/100 [03:00<00:00,  1.80s/it]


Epoch 49/50, Loss: 69.3352


100%|██████████| 100/100 [02:59<00:00,  1.80s/it]


Epoch 50/50, Loss: 69.4092


# Baseline training on trimmed_fan data from MIMII dataset

In [9]:
# ----------------------------
# Config
# ----------------------------

class Config:
    SAMPLE_RATE = 16000
    MODEL_NAME = "facebook/hubert-base-ls960"
    BATCH_SIZE = 8
    EPOCHS = 50
    LR = 1e-4
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    DATA_PATH = Path("data/trimmed_fan")

config = Config()

# ----------------------------
# Dataset
# ----------------------------
class AudioDataset(Dataset):
    def __init__(self, data_path):
        self.filepaths = []
        self.labels = []

        for label in ["normal", "abnormal"]:
            for id in ["id_00", "id_02", "id_04", "id_06"]:
                folder = data_path / label / id
                for file in folder.glob("*.wav"):
                    self.filepaths.append(file)
                    self.labels.append(label)

        self.label_encoder = LabelEncoder()
        self.encoded_labels = self.label_encoder.fit_transform(self.labels)

        # self.processor = Wav2Vec2Processor.from_pretrained(config.MODEL_NAME)

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        path = self.filepaths[idx]
        try:
            # Convert Path object to string
            path_str = str(path)
            waveform, sr = torchaudio.load(path_str)
            waveform = torchaudio.functional.resample(waveform, sr, config.SAMPLE_RATE)
            input_values = waveform.squeeze(0)  # Use raw waveform for HuBERT
            label = torch.tensor(self.encoded_labels[idx], dtype=torch.long)
            return input_values, label
        except Exception as e:
            print(f"Error loading {path}: {e}")
            # Return a default fallback or raise the error
            raise

In [10]:
# ----------------------------
# Transformer Model
# ----------------------------
class HubertClassifier(nn.Module):
    def __init__(self, hidden_dim=768, num_classes=2):
        super(HubertClassifier, self).__init__()
        self.hubert = HubertModel.from_pretrained(config.MODEL_NAME)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes)
        )

    def forward(self, input_values):
        with torch.no_grad():
            features = self.hubert(input_values).last_hidden_state
        pooled = torch.mean(features, dim=1)
        logits = self.classifier(pooled)
        return logits

# ----------------------------
# Training Loop
# ----------------------------
def train():
    dataset = AudioDataset(config.DATA_PATH)
    dataloader = DataLoader(dataset, batch_size=config.BATCH_SIZE, shuffle=True)

    model = HubertClassifier().to(config.DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.classifier.parameters(), lr=config.LR)

    model.train()
    for epoch in range(config.EPOCHS):
        running_loss = 0.0
        for inputs, labels in tqdm(dataloader):
            inputs = inputs.to(config.DEVICE)
            labels = labels.to(config.DEVICE)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f"Epoch {epoch + 1}/{config.EPOCHS}, Loss: {running_loss:.4f}")

    torch.save(model.state_dict(), "hubert_transformer_trimmed_fan_bs8_lr4.pth")

# ----------------------------
# Run
# ----------------------------
if __name__ == '__main__':
    train()

100%|██████████| 30/30 [00:48<00:00,  1.62s/it]


Epoch 1/50, Loss: 21.0570


100%|██████████| 30/30 [00:49<00:00,  1.64s/it]


Epoch 2/50, Loss: 20.7244


100%|██████████| 30/30 [00:48<00:00,  1.63s/it]


...

Epoch 48/50, Loss: 18.1018


100%|██████████| 30/30 [00:53<00:00,  1.79s/it]


Epoch 49/50, Loss: 17.9130


100%|██████████| 30/30 [00:48<00:00,  1.62s/it]


Epoch 50/50, Loss: 18.3031


# Attempting to address overfitting with an L2 norm regularizer

In [7]:

# ----------------------------
# Config
# ----------------------------

class Config:
    SAMPLE_RATE = 16000
    MODEL_NAME = "facebook/hubert-base-ls960"
    BATCH_SIZE = 8
    EPOCHS = 50
    LR = 1e-4
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    DATA_PATH = Path("data/trimmed_fan")

config = Config()

# ----------------------------
# Dataset
# ----------------------------
class AudioDataset(Dataset):
    def __init__(self, data_path):
        self.filepaths = []
        self.labels = []

        for label in ["normal", "abnormal"]:
            for id in ["id_00", "id_02", "id_04", "id_06"]:
                folder = data_path / label / id
                for file in folder.glob("*.wav"):
                    self.filepaths.append(file)
                    self.labels.append(label)

        self.label_encoder = LabelEncoder()
        self.encoded_labels = self.label_encoder.fit_transform(self.labels)

        # self.processor = Wav2Vec2Processor.from_pretrained(config.MODEL_NAME)

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        path = self.filepaths[idx]
        try:
            # Convert Path object to string
            path_str = str(path)
            waveform, sr = torchaudio.load(path_str)
            waveform = torchaudio.functional.resample(waveform, sr, config.SAMPLE_RATE)
            input_values = waveform.squeeze(0)  # Use raw waveform for HuBERT
            label = torch.tensor(self.encoded_labels[idx], dtype=torch.long)
            return input_values, label
        except Exception as e:
            print(f"Error loading {path}: {e}")
            # Return a default fallback or raise the error
            raise

In [10]:
class HubertClassifier(nn.Module):
    def __init__(self, hidden_dim=768, num_classes=2, dropout_rate=0.5):
        super(HubertClassifier, self).__init__()
        self.hubert = HubertModel.from_pretrained(config.MODEL_NAME)
        # Freeze HubertModel parameters initially
        for param in self.hubert.parameters():
            param.requires_grad = False
            
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, num_classes)
        )

    def forward(self, input_values):
        features = self.hubert(input_values).last_hidden_state
        pooled = torch.mean(features, dim=1)
        logits = self.classifier(pooled)
        return logits

def train():
    # Create train/validation split
    full_dataset = AudioDataset(config.DATA_PATH)
    train_size = int(0.8 * len(full_dataset))
    val_size = len(full_dataset) - train_size
    train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])
    
    train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE)

    model = HubertClassifier(dropout_rate=0.5).to(config.DEVICE)
    criterion = nn.CrossEntropyLoss()
    
    # Add significant weight decay (L2 regularization)
    optimizer = torch.optim.AdamW(model.classifier.parameters(), 
                                 lr=config.LR, 
                                 weight_decay=0.01)  # Strong L2 regularization
    
    # Add learning rate scheduler
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=2, verbose=True
    )
    
    best_val_loss = float('inf')
    
    for epoch in range(config.EPOCHS):
        # Training phase
        model.train()
        running_loss = 0.0
        for inputs, labels in tqdm(train_loader):
            inputs = inputs.to(config.DEVICE)
            labels = labels.to(config.DEVICE)

            optimizer.zero_grad()
            outputs = model(inputs)
            
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        train_loss = running_loss / len(train_loader)
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = inputs.to(config.DEVICE)
                labels = labels.to(config.DEVICE)
                
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        
        val_loss = val_loss / len(val_loader)
        val_accuracy = 100 * correct / total
        
        print(f"Epoch {epoch + 1}/{config.EPOCHS}")
        print(f"Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy:.2f}%")
        
        # Learning rate scheduling
        scheduler.step(val_loss)
        
        # Save best model (without early stopping)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), "hubert_classifier_best.pth")
        
        # Unfreeze HubertModel layers gradually (after 10 epochs)
        if epoch == 10:
            print("Unfreezing HubertModel layers...")
            for param in model.hubert.encoder.layer[-2:].parameters():
                param.requires_grad = True
            
            # Update optimizer to include unfrozen parameters
            # Keep strong L2 regularization (weight decay)
            optimizer = torch.optim.AdamW([
                {'params': model.classifier.parameters(), 'weight_decay': 0.01},
                {'params': model.hubert.encoder.layer[-2:].parameters(), 'lr': config.LR/10, 'weight_decay': 0.001}
            ], lr=config.LR)
            
            # Reset scheduler with new optimizer
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, mode='min', factor=0.5, patience=2, verbose=True
            )

    # Save the final model
    torch.save(model.state_dict(), "hubert_transformer_final.pth")

# ----------------------------
# Run
# ----------------------------
if __name__ == '__main__':
    train()

/opt/miniconda3/envs/torchpy311/lib/python3.11/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
100%|██████████| 24/24 [01:09<00:00,  2.91s/it]


Epoch 1/50
Train Loss: 0.7677, Val Loss: 0.6735, Val Accuracy: 60.42%


100%|██████████| 24/24 [01:04<00:00,  2.67s/it]


Epoch 2/50
Train Loss: 0.7385, Val Loss: 0.6535, Val Accuracy: 58.33%


100%|██████████| 24/24 [01:11<00:00,  3.00s/it]


...

100%|██████████| 24/24 [01:15<00:00,  3.16s/it]


Epoch 10/50
Train Loss: 0.6231, Val Loss: 0.6199, Val Accuracy: 64.58%


100%|██████████| 24/24 [01:17<00:00,  3.22s/it]


Epoch 11/50
Train Loss: 0.6094, Val Loss: 0.6469, Val Accuracy: 64.58%
Unfreezing HubertModel layers...


AttributeError: 'HubertEncoder' object has no attribute 'layer'